The following instructions are to set up Langfuse for tracing your agents. This is optional but highly recommended for debugging and understanding agent behavior. Assuming that you have Docker installed, follow these steps:

```bash
git clone https://github.com/langfuse/langfuse.git
cd langfuse
docker compose up -d
```

Go to http://localhost:3000 and do this:

- Sign up.
- Create an organization.
- Create a project.
- Create an API key.

Insert the API key into your environment by adding the following line to your `.env` file:

```bash
# LangFuse.
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_HOST=http://localhost:3000
```

## Import environment variables

In [3]:
import os
import dotenv
dotenv.load_dotenv()
print("Environment variables loaded.")

Environment variables loaded.


## Create a callback handler

In [4]:
from langfuse import Langfuse

langfuse = Langfuse(
  secret_key=os.environ["LANGFUSE_SECRET_KEY"],
  public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
  host=os.environ["LANGFUSE_HOST"],
)

Exception while exporting Span.
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/jupyterlab/4.3.3/libexec/lib/python3.13/site-packages/urllib3/connection.py", line 199, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/opt/homebrew/Cellar/jupyterlab/4.3.3/libexec/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/opt/homebrew/Cellar/jupyterlab/4.3.3/libexec/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/homebrew/Cellar/jupyterlab/4.3.3/libexec/lib/python3.13/site-packages/urllib3/connectionpool.py", line 789, in urlopen
    response = self._ma

In [5]:
from langfuse.langchain import CallbackHandler

# Load environment variables from .env file.
dotenv.load_dotenv()

# Initialize the Langfuse handler
langfuse_handler = CallbackHandler()

## Run the agent

We will run the same ReAct agent as before, but this time with Langfuse tracing enabled.

In [6]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

# Initialize the chat model.
model = init_chat_model(
    os.environ["LANGCHAIN_CHAT_MODEL_ANTHROPIC"],
)

# Create the react agent without any tools.
agent = create_react_agent(
    model=model,
    tools=[],
)

# Invoke the agent with a user message.
result = agent.invoke(
    { "messages": [
        {
            "role": "user",
            "content": "Should I use LLM-powered agents or agentic workflows in 2025?",
        }
    ]},
    config={"callbacks": [langfuse_handler]}
)
print(result["messages"][-1].content)

/opt/homebrew/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/x0/xs9r1crn50s7y70ntqd5wyvr0000gn/T/ipykernel_36850/531836785.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


# LLM Agents vs. Agentic Workflows in 2025

The honest answer: **it depends on your specific problem**, but here's how to think about it:

## Use LLM-powered agents when:
- **Unpredictability is high** — customer support, research, exploration
- **You need real-time adaptation** — the agent encounters novel situations regularly
- **Tool use is complex and varied** — deciding *which* tool to use matters as much as using it
- **You can tolerate occasional failures** — errors are recoverable or low-stakes
- **You want minimal upfront engineering** — faster to prototype

**Trade-off**: Less predictable, potentially higher latency, can be expensive at scale

## Use agentic workflows when:
- **The process is relatively structured** — you know the major steps involved
- **Reliability is critical** — failures are costly
- **You need consistent performance** — repeatable, auditable execution
- **Latency matters** — you need deterministic timing
- **Cost is a concern** — fewer LLM calls per task

Now navigate to your Langfuse dashboard to see the traces of your agent's interactions.

# Done.